In [ ]:
# Part 1: Data Loading & FC Matrix (n=150)
from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker
from nilearn.datasets import fetch_atlas_harvard_oxford
from sklearn.svm import SVC, SVR
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.utils import shuffle
import numpy as np
import pandas as pd

# --- Step 1. Data Loading ---
num_subjects = 150
data = datasets.fetch_development_fmri(n_subjects=num_subjects, age_group='both', verbose=0)
atlas = fetch_atlas_harvard_oxford('cort-maxprob-thr25-2mm')
masker = NiftiLabelsMasker(labels_img=atlas.maps, standardize=True,
                           memory='nilearn_cache', verbose=0)

# --- Step 2. FC Matrix Computation ---
fc_matrices = []
for func_file in data.func:
    time_series = masker.fit_transform(func_file)
    fc_matrices.append(np.corrcoef(time_series.T))
fc_matrices = np.array(fc_matrices)

# --- Step 3. Flatten + NaN Handling ---
X = fc_matrices.reshape(len(fc_matrices), -1)
X = np.nan_to_num(X, nan=0.0)

# --- Step 4. Labels ---
df = pd.DataFrame(data.phenotypic)
y = (df['Child_Adult'] == 'adult').astype(int).values
y_age = df['Age'].astype(float).values
X, y, y_age = shuffle(X, y, y_age, random_state=42)

# --- Step 5. Classification (child/adult) ---
pipe_cls = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear', class_weight='balanced', random_state=42))
])
cv_cls = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe_cls, X, y, cv=cv_cls, scoring='accuracy')
print(f"=== Classification (n={len(X)}) ===")
print(f"Accuracy per fold: {scores}")
print(f"Mean accuracy: {scores.mean():.3f} ± {scores.std():.3f}")
y_pred = cross_val_predict(pipe_cls, X, y, cv=cv_cls)
print(classification_report(y, y_pred, target_names=['child', 'adult']))

# --- Step 6. Age Regression (full cohort) ---
pipe_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='linear'))
])
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = -cross_val_score(pipe_reg, X, y_age, cv=cv_reg,
                               scoring='neg_mean_absolute_error')
print(f"=== Age Regression (n={len(X)}) ===")
print(f"MAE per fold: {mae_scores}")
print(f"Mean MAE: {mae_scores.mean():.2f} yrs ± {mae_scores.std():.2f} yrs")

In [ ]:
# Part 2: Child Filtering & Upper-Triangle Features
import matplotlib.pyplot as plt
from sklearn.utils import shuffle as sk_shuffle

# --- Step 1. Reproduce Shuffle Index & Filter Children ---
df_full = pd.DataFrame(data.phenotypic)
_, _, _, df_idx = sk_shuffle(X, y, y_age,
                              pd.Series(range(len(df_full))),
                              random_state=42)
df_shuffled = df_full.iloc[df_idx.values].reset_index(drop=True)

child_mask = df_shuffled['Child_Adult'] == 'child'
X_child = X[child_mask]
y_child_age = y_age[child_mask]

print(f"Number of children : {child_mask.sum()}")
print(f"Age range          : {y_child_age.min():.1f} – {y_child_age.max():.1f} yrs")
print(f"X_child shape      : {X_child.shape}")

# --- Step 2. Upper-Triangle Flatten (2304 → 1128 features) ---
n_rois = 48
triu_idx = np.triu_indices(n_rois, k=1)

fc_matrices_child = X_child.reshape(-1, n_rois, n_rois)
X_child_triu = np.array([m[triu_idx] for m in fc_matrices_child])
X_child_triu = np.nan_to_num(X_child_triu, nan=0.0)

print(f"X_child_triu shape : {X_child_triu.shape}")

# --- Step 3. Baseline SVR — Pediatric Brain Age ---
pipe_child = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='linear'))
])
cv_child = KFold(n_splits=5, shuffle=True, random_state=42)
mae_child = -cross_val_score(pipe_child, X_child_triu, y_child_age,
                              cv=cv_child,
                              scoring='neg_mean_absolute_error')

print(f"\n=== Pediatric Brain Age — SVR Baseline (n={child_mask.sum()}) ===")
print(f"MAE per fold : {mae_child}")
print(f"Mean MAE     : {mae_child.mean():.2f} yrs ± {mae_child.std():.2f} yrs")

# --- Step 4. Brain Age Gap Scatter Plot (SVR) ---
y_pred_child = cross_val_predict(pipe_child, X_child_triu, y_child_age,
                                  cv=cv_child)
gap = y_pred_child - y_child_age

plt.figure(figsize=(7, 6))
plt.scatter(y_child_age, y_pred_child, alpha=0.7, color='steelblue')
plt.plot([3, 13], [3, 13], 'r--', label='Perfect prediction')
plt.xlabel('Actual age (yrs)')
plt.ylabel('Predicted age (yrs)')
plt.title(f'Pediatric Brain Age — SVR\nMAE = {mae_child.mean():.2f} yrs')
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nBrain Age Gap mean : {gap.mean():.2f} yrs")
print(f"Brain Age Gap std  : {gap.std():.2f} yrs")

In [ ]:
# Part 3: Model Comparison — Full Cohort (n=150)
from sklearn.linear_model import Ridge, ElasticNet, LogisticRegression
from sklearn.svm import SVC, SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
import numpy as np

# --- Step 1. Upper-Triangle Flatten for Full Cohort ---
n_rois = 48
triu_idx = np.triu_indices(n_rois, k=1)

X_triu = np.array([
    row.reshape(n_rois, n_rois)[triu_idx]
    for row in X
])
X_triu = np.nan_to_num(X_triu, nan=0.0)

print(f"X_triu shape: {X_triu.shape}")

# --- Step 2. Age Regression Model Comparison ---
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

models_reg = {
    'SVR (linear)' : Pipeline([('scaler', StandardScaler()), ('model', SVR(kernel='linear'))]),
    'Ridge'        : Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'ElasticNet'   : Pipeline([('scaler', StandardScaler()), ('model', ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000))]),
}

print(f"=== Age Regression Model Comparison (n={len(X_triu)}) ===")
reg_results = {}
for name, pipe in models_reg.items():
    mae = -cross_val_score(pipe, X_triu, y_age, cv=cv_reg,
                           scoring='neg_mean_absolute_error')
    r2  =  cross_val_score(pipe, X_triu, y_age, cv=cv_reg,
                           scoring='r2')
    reg_results[name] = {'MAE': mae.mean(), 'MAE_std': mae.std(), 'R2': r2.mean()}
    print(f"{name:15s} | MAE: {mae.mean():.2f} ± {mae.std():.2f} yrs | R²: {r2.mean():.3f}")

# --- Step 3. Classification Model Comparison ---
cv_cls = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_cls = {
    'SVM (linear)'       : Pipeline([('scaler', StandardScaler()), ('model', SVC(kernel='linear', class_weight='balanced', random_state=42))]),
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))]),
}

print(f"\n=== Classification Model Comparison (n={len(X_triu)}) ===")
for name, pipe in models_cls.items():
    acc = cross_val_score(pipe, X_triu, y, cv=cv_cls, scoring='accuracy')
    f1  = cross_val_score(pipe, X_triu, y, cv=cv_cls, scoring='f1')
    print(f"{name:22s} | Acc: {acc.mean():.3f} ± {acc.std():.3f} | F1: {f1.mean():.3f}")

In [ ]:
# Part 4: Model Comparison — Pediatric Brain Age (n=118)
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_score, permutation_test_score
import numpy as np

# --- Step 1. Model Comparison ---
cv_child = KFold(n_splits=5, shuffle=True, random_state=42)

models_child = {
    'SVR (linear)' : Pipeline([('scaler', StandardScaler()),
                                ('model', SVR(kernel='linear'))]),
    'ElasticNet'   : Pipeline([('scaler', StandardScaler()),
                                ('model', ElasticNet(alpha=0.1, l1_ratio=0.5,
                                                     max_iter=5000))]),
    'KRR (RBF)'    : Pipeline([('scaler', StandardScaler()),
                                ('model', KernelRidge(kernel='rbf', alpha=1.0))]),
    'PCA+Ridge'    : Pipeline([('scaler', StandardScaler()),
                                ('pca',   PCA(n_components=50)),
                                ('model', Ridge(alpha=1.0))]),
}

print(f"=== Pediatric Brain Age Model Comparison (n={len(X_child_triu)}) ===")
results = {}
for name, pipe in models_child.items():
    mae = -cross_val_score(pipe, X_child_triu, y_child_age,
                           cv=cv_child, scoring='neg_mean_absolute_error')
    r2  =  cross_val_score(pipe, X_child_triu, y_child_age,
                           cv=cv_child, scoring='r2')
    results[name] = mae.mean()
    print(f"{name:15s} | MAE: {mae.mean():.2f} ± {mae.std():.2f} yrs | R²: {r2.mean():.3f}")

# --- Step 2. Permutation Test (best model) ---
best_name = min(results, key=results.get)
best_pipe = models_child[best_name]

print(f"\n=== Permutation Test — {best_name} ===")
score, perm_scores, p_value = permutation_test_score(
    best_pipe, X_child_triu, y_child_age,
    cv=cv_child,
    scoring='neg_mean_absolute_error',
    n_permutations=100,
    random_state=42,
    n_jobs=1
)

print(f"Observed MAE : {-score:.2f} yrs")
print(f"Random MAE   : {-perm_scores.mean():.2f} yrs ± {perm_scores.std():.2f} yrs")
print(f"p-value      : {p_value:.3f}")
print(f"Significant  : {'Yes (p < 0.05)' if p_value < 0.05 else 'No'}")

In [ ]:
# Part 5: Final Model & Brain Age Report
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

# --- Step 1. Train Final Model (PCA + Ridge, n=118) ---
pipe_final = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=50)),
    ('model',  Ridge(alpha=1.0))
])
pipe_final.fit(X_child_triu, y_child_age)

y_pred_all = pipe_final.predict(X_child_triu)
gap_all = y_pred_all - y_child_age

# --- Step 2. Individual Brain Age Report Function ---
def brain_age_report(subject_idx):
    """
    Generate a Brain Age Report for a single subject.
    subject_idx: int, 0 to 117 (index within child dataset)
    """
    actual_age    = y_child_age[subject_idx]
    predicted_age = y_pred_all[subject_idx]
    gap           = predicted_age - actual_age

    # Age-band MAE (manually set based on observed fold errors)
    age_bands = [(3, 6, 2.1), (6, 9, 1.4), (9, 13, 1.2)]
    age_band_mae = next((mae for lo, hi, mae in age_bands
                         if lo <= actual_age < hi), 1.59)

    # Peer group: children within ±1.5 yrs, excluding self
    peer_mask = (
        (np.abs(y_child_age - actual_age) <= 1.5) &
        (np.arange(len(y_child_age)) != subject_idx)
    )
    peer_gaps  = gap_all[peer_mask]
    percentile = (peer_gaps < gap).mean() * 100

    # Interpretation
    if gap > 0.5:
        interpretation = f"Brain development approximately {gap:.1f} yrs ahead of peers"
    elif gap < -0.5:
        interpretation = f"Brain development approximately {abs(gap):.1f} yrs behind peers"
    else:
        interpretation = "Brain development within typical range for peers"

    # --- Plot ---
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Left: peer gap distribution
    axes[0].hist(peer_gaps, bins=15, color='lightsteelblue',
                 edgecolor='white', alpha=0.85, label='Peer Brain Age Gap')
    axes[0].axvline(gap, color='crimson', linewidth=2.5,
                    label=f'This child: gap = {gap:+.2f} yrs')
    axes[0].axvline(0, color='gray', linewidth=1, linestyle='--',
                    label='Reference (gap = 0)')
    axes[0].set_xlabel('Brain Age Gap (predicted − actual age, yrs)')
    axes[0].set_ylabel('Number of peers')
    axes[0].set_title(
        f'Position within peer distribution\n'
        f'(age {actual_age:.1f} yrs ± 1.5 yrs, n = {peer_mask.sum()} peers)'
    )
    axes[0].legend(fontsize=9)

    # Right: predicted vs actual scatter
    axes[1].scatter(y_child_age, y_pred_all, alpha=0.4,
                    color='steelblue', s=40, label='All children')
    axes[1].scatter(actual_age, predicted_age, color='crimson',
                    s=150, zorder=5, label=f'This child (idx={subject_idx})')
    axes[1].errorbar(actual_age, predicted_age,
                     yerr=age_band_mae, fmt='none',
                     color='crimson', capsize=6, linewidth=2)
    axes[1].plot([3, 13], [3, 13], 'k--', linewidth=1, label='Perfect prediction')
    axes[1].set_xlabel('Actual age (yrs)')
    axes[1].set_ylabel('Predicted age (yrs)')
    axes[1].set_title('Brain Age Scatter — All Children')
    axes[1].legend(fontsize=9)

    plt.suptitle(
        f'Brain Age Report  |  Actual: {actual_age:.1f} yrs  |  '
        f'Predicted: {predicted_age:.1f} yrs  |  Gap: {gap:+.2f} yrs\n'
        f'→ {interpretation}  (Top {100-percentile:.0f}% within peers)',
        fontsize=11, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.show()

    rank_in_peers = int((1 - percentile / 100) * peer_mask.sum()) + 1
    print(f"Actual age      : {actual_age:.1f} yrs")
    print(f"Predicted age   : {predicted_age:.1f} yrs  (± {age_band_mae:.2f} yrs)")
    print(f"Brain Age Gap   : {gap:+.2f} yrs")
    print(f"Peer ranking    : Top {100-percentile:.0f}%  "
          f"({rank_in_peers} / {peer_mask.sum()} age-matched peers)")
    print(f"Interpretation  : {interpretation}")

# --- Step 3. Run Report ---
brain_age_report(subject_idx=69)